# DP-MLM: replication progress & new datasets
**COMP8240 project update** · Bayarbaatar Gonchigsuren (48728497)

Replicating *Meisenbacher et al. (2024), DP-MLM: Differentially Private Text Rewriting Using Masked Language Models*, Findings of ACL 2024 ([code](https://github.com/sjmeis/DPMLM), PyPI package `dpmlm`).

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
With `QUICK = True` the whole notebook takes roughly 45–70 minutes on a T4. Every table and figure is saved to `results/` and zipped into `results.zip` at the end.

| Part | What it shows |
|---|---|
| 1 | The official DP-MLM code running live on example sentences at ε ∈ {10, 25, 50, 100, 250} |
| 2 | Replication on original data (GLUE SST-2): BLEU, embedding cosine similarity and sentiment utility vs ε |
| 3 | **Existing new dataset:** IMDb movie reviews |
| 4 | **Constructed new dataset:** Reddit posts with self-disclosed age and gender (collection → cleaning → labels → audit sample) |
| 5 | Privacy experiment on the constructed dataset: static and adaptive attackers, PP+ and Relative Gain |
| 6 | LLM-as-judge (meaning + fluency of rewrites), LLM label annotator (Cohen's κ) and zero-shot LLM attacker |
| 7 | Export: every number, table and figure used in the final report is written to `results/generated/` |

In [ ]:
# --- Install (about 1-2 min) ---
!pip install -q dpmlm sentence-transformers sacrebleu datasets scikit-learn spacy tabulate accelerate
!python -m spacy download en_core_web_sm -q

In [ ]:
# --- NLTK resources + a small compatibility patch for the dpmlm package ---
import nltk
for p in ["stopwords", "wordnet", "punkt", "punkt_tab", "words",
          "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(p, quiet=True)

# dpmlm 1.3.0 sets torch.backends.cuda.matmul.fp32_precision at import time, which only
# exists in very recent PyTorch. Wrap it in try/except so the package imports on Colab's torch.
import importlib.util, pathlib
core = pathlib.Path(importlib.util.find_spec("dpmlm").origin).parent / "core.py"
src = core.read_text()
line = "torch.backends.cuda.matmul.fp32_precision = 'tf32'"
if line in src and "fp32_patch" not in src:
    src = src.replace(line, "try:  # fp32_patch\n    " + line + "\nexcept Exception:\n    pass")
    core.write_text(src)
print("dpmlm core:", core)

In [ ]:
# --- Configuration ---
import os, re, time, json, random, warnings, zipfile
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)

QUICK = True                      # False = bigger samples (roughly 3x the runtime)
EPSILONS = [10, 25, 50, 100, 250]  # per-token privacy budgets used in the paper
N_SST2 = 150 if QUICK else 500     # SST-2 validation sentences to privatise
N_IMDB = 150 if QUICK else 500     # IMDb reviews to privatise
N_REDDIT = 4000 if QUICK else 8000 # size of the balanced constructed-dataset sample
N_PRIV_TEST = 200 if QUICK else 400   # held-out Reddit test texts (privatised at every eps)
N_PRIV_ADAPT = 400 if QUICK else 1000 # Reddit training texts privatised for the adaptive attacker
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

OUT = "results"; os.makedirs(OUT, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU - switch runtime to T4!")

## Part 1 · Run DP-MLM (official code)
**Architecture recap.** For each non-stopword token in the input: (1) mask that token, (2) concatenate the *original* sentence with the masked sentence, (3) run RoBERTa-base to get logits for the masked position, (4) clip logits to [−3.21, 16.30] (sensitivity Δ ≈ 19.5), (5) scale by ε / 2Δ and softmax, which is the exponential mechanism, (6) sample the replacement token. Doing this for n tokens costs nε in total under sequential composition.

In [ ]:
from dpmlm import DPMLM
t0 = time.time()
M = DPMLM()   # FacebookAI/roberta-base with the paper's default clipping bounds
print(f"Loaded in {time.time()-t0:.1f}s | clip=[{M.clip_min:.3f}, {M.clip_max:.3f}] | sensitivity={M.sensitivity:.3f}")

def privatize(text, eps):
    text = " ".join(str(text).split())
    if not text:
        return text
    out = M.dpmlm_rewrite_batch(text, epsilon=eps, batch_size=64)
    return out[0] if isinstance(out, tuple) else out

def privatize_many(texts, eps, desc=""):
    return [privatize(t, eps) for t in tqdm(list(texts), desc=f"{desc} eps={eps}", leave=False)]

In [ ]:
demo = ["John lives in Sydney and works at a bank.",
        "The food was cold and the waiter was rude, so we are never coming back.",
        "I am a 24 year old nurse and my boyfriend is a high school teacher."]
rows = [{"original": s, "epsilon": e, "rewritten": privatize(s, e)} for s in demo for e in EPSILONS]
demo_df = pd.DataFrame(rows)
demo_df.to_csv(f"{OUT}/demo_rewrites.csv", index=False)
for s in demo:
    print("\nORIGINAL :", s)
    for _, r in demo_df[demo_df.original == s].iterrows():
        print(f"  eps={r.epsilon:<4} {r.rewritten}")

t = time.time(); _ = [privatize(demo[1], 50) for _ in range(5)]
print(f"\nSpeed: ~{(time.time()-t)/5:.2f}s per sentence on {DEVICE}")

### Shared evaluation code
* **BLEU** (sacreBLEU, corpus level, 0–100): word overlap with the original. Lower means more rewriting.
* **CosSim**: mean cosine similarity of `all-MiniLM-L6-v2` sentence embeddings (as in the paper). Shows how much meaning is kept.
* **Acc**: accuracy of an off-the-shelf SST-2 sentiment classifier on the rewritten text. This is a cheap *static* utility proxy. The paper instead fine-tunes DeBERTaV3 on privatised training data; that is the next step.

In [ ]:
import sacrebleu
from sentence_transformers import SentenceTransformer
from transformers import pipeline

sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
sent_clf = pipeline("text-classification",
                    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
                    device=0 if DEVICE == "cuda" else -1)

def bleu(orig, priv):
    return sacrebleu.corpus_bleu(list(priv), [list(orig)]).score

def cossim(orig, priv):
    a = sbert.encode(list(orig), convert_to_tensor=True, normalize_embeddings=True, batch_size=64)
    b = sbert.encode(list(priv), convert_to_tensor=True, normalize_embeddings=True, batch_size=64)
    return float((a * b).sum(1).mean())

def sent_acc(texts, labels):
    preds = [1 if p["label"] == "POSITIVE" else 0 for p in sent_clf(list(texts), batch_size=64, truncation=True)]
    return float(np.mean(np.array(preds) == np.array(labels)))

def changed_rate(orig, priv):
    rates = []
    for o, p in zip(orig, priv):
        ow, pw = o.lower().split(), p.lower().split()
        n = max(len(ow), 1)
        rates.append(sum(1 for x, y in zip(ow, pw) if x != y) / n)
    return float(np.mean(rates))

def eval_utility(texts, labels, name):
    texts, labels = list(texts), list(labels)
    rows = [{"dataset": name, "epsilon": "original", "BLEU": 100.0, "CosSim": 1.0,
             "Acc": sent_acc(texts, labels), "%words changed": 0.0}]
    priv = {}
    for e in EPSILONS:
        p = privatize_many(texts, e, name); priv[e] = p
        rows.append({"dataset": name, "epsilon": e, "BLEU": bleu(texts, p), "CosSim": cossim(texts, p),
                     "Acc": sent_acc(p, labels), "%words changed": 100 * changed_rate(texts, p)})
    res = pd.DataFrame(rows).round(3)
    long = pd.DataFrame([{"id": i, "epsilon": e, "original": texts[i], "label": labels[i], "privatized": priv[e][i]}
                         for e in EPSILONS for i in range(len(texts))])
    tag = name.lower().replace("-", "")
    res.to_csv(f"{OUT}/{tag}_utility.csv", index=False)
    long.to_csv(f"{OUT}/{tag}_privatized.csv", index=False)
    return res, priv

def plot_utility(results, fname):
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
    for ax, metric in zip(axes, ["BLEU", "CosSim", "Acc"]):
        for name, res in results.items():
            r = res[res.epsilon != "original"]
            ax.plot(r.epsilon.astype(float), r[metric], marker="o", label=name)
            ax.axhline(res[res.epsilon == "original"][metric].iloc[0], ls="--", lw=0.8,
                       color=ax.lines[-1].get_color())
        ax.set_xscale("log"); ax.set_xticks(EPSILONS); ax.set_xticklabels(EPSILONS)
        ax.set_xlabel("ε (per token)"); ax.set_title(metric); ax.grid(alpha=0.3)
    axes[0].legend(); plt.tight_layout(); plt.savefig(f"{OUT}/{fname}", dpi=160); plt.show()

## Part 2 · Replication on original data: GLUE SST-2
SST-2 is one of the GLUE tasks used in the paper (and one of the four on which the authors report BLEU and cosine similarity). Dashed lines are the non-private baseline.

In [ ]:
from datasets import load_dataset
sst = load_dataset("nyu-mll/glue", "sst2", split="validation").shuffle(seed=SEED).select(range(N_SST2))
print(f"SST-2 sample: {len(sst)} sentences | mean length {np.mean([len(s.split()) for s in sst['sentence']]):.1f} words | "
      f"positive share {np.mean(sst['label']):.2f}")
t = time.time()
sst_res, sst_priv = eval_utility(sst["sentence"], sst["label"], "SST-2")
print(f"done in {(time.time()-t)/60:.1f} min")
display(sst_res)
for i in range(3):
    print("\nORIG :", sst["sentence"][i])
    for e in EPSILONS: print(f"  eps={e:<4}", sst_priv[e][i])

## Part 3 · Existing new dataset: IMDb movie reviews
50,000 reviews with binary sentiment labels (HuggingFace `stanfordnlp/imdb`); DP-MLM has not been evaluated on it. Reviews average about 230 words, so to keep cost manageable (privacy cost is nε for n tokens) each review is cut to its leading sentences, at most 60 words. The baseline is measured on the same cut text.

In [ ]:
def first_words(text, max_words=60):
    text = " ".join(re.sub(r"<br\s*/?>", " ", text).split())
    sents = re.split(r"(?<=[.!?])\s+", text)
    out = []
    for s in sents:
        if len(" ".join(out + [s]).split()) > max_words:
            break
        out.append(s)
    return " ".join(out) if out else " ".join(text.split()[:max_words])

imdb = load_dataset("stanfordnlp/imdb", split="test").shuffle(seed=SEED).select(range(N_IMDB))
imdb_full_len = [len(t.split()) for t in imdb["text"]]
imdb_text = [first_words(t) for t in imdb["text"]]
print(f"IMDb sample: {len(imdb_text)} reviews | full review {np.mean(imdb_full_len):.0f} words on average -> "
      f"truncated {np.mean([len(t.split()) for t in imdb_text]):.0f} words | positive share {np.mean(imdb['label']):.2f}")
t = time.time()
imdb_res, imdb_priv = eval_utility(imdb_text, imdb["label"], "IMDb")
print(f"done in {(time.time()-t)/60:.1f} min")
display(imdb_res)
plot_utility({"SST-2": sst_res, "IMDb": imdb_res}, "utility_sst2_imdb.png")

## Part 4 · Constructed dataset: Reddit self-disclosure (age bracket + gender)
This plays the same role as Trustpilot (gender) and Yelp (author) in the paper, in a new domain, with **new sensitive attributes built from self-disclosure**.

**Source.** Reddit posts from the Reddit TL;DR corpus (Völske et al., 2017), as filtered by Stiennon et al. (2020) (`trl-lib/tldr` on HuggingFace, open licence). Advice subreddits follow a convention where posters tag themselves in the title, e.g. *"Me [24 F] with my boyfriend [27 M] of 2 years…"*.

**Pipeline.**
1. **Collection**: parse subreddit, title and post for every example.
2. **Label extraction (regex)**: keep posts whose title *starts* with the poster's own tag (`Me [24F]`, `I (31 m)`, …). Labels: **gender** (M/F) and **age bracket** (≤25 vs >25).
3. **Cleaning / leakage removal**: remove every age/gender tag from the post text, drop posts that state the poster's age in words (*"I'm 24"*, *"24 years old"*), replace URLs, `u/` usernames and person names (spaCy NER) with placeholders, and cut to 15–60 words.
4. **Validation**: check the title label against a tag in the post body where one exists, and export a 100-example **manual audit** file.
5. **Balanced sample** over gender × age bracket for the experiments.

In [ ]:
# ---------- Constructed dataset: parsing + label extraction ----------
PROMPT_RE = re.compile(r"SUBREDDIT:\s*r/(?P<sub>\S+)\s*TITLE:\s*(?P<title>.*?)\s*POST:\s*(?P<post>.*?)\s*(?:TL;DR:.*)?$", re.S)
_TAG_BODY = (r"(?:(?P<a1>1[3-9]|[2-6]\d)\s*[/,\- ]?\s*(?P<g1>m|f|male|female)"
             r"|(?P<g2>m|f|male|female)\s*[/,\- ]?\s*(?P<a2>1[3-9]|[2-6]\d))")
_TAG = r"[\[\(\{]\s*" + _TAG_BODY + r"\s*[\]\)\}]"
POSTER_TITLE_RE = re.compile(r"^\W*(?:me|i|i'm|im|i am|myself)\s*,?\s*" + _TAG, re.I)
POSTER_POST_RE  = re.compile(r"\b(?:me|i|i'm|im|i am|myself)\s*,?\s*" + _TAG, re.I)
ANY_TAG_RE  = re.compile(_TAG, re.I)
BARE_TAG_RE = re.compile(r"\b(?:(?:1[3-9]|[2-6]\d)(?:m|f)|(?:m|f)(?:1[3-9]|[2-6]\d))\b", re.I)
EXPLICIT_AGE_RE = re.compile(r"\b(?:i'?m|i am|im)\s+(?:a\s+)?(?:1[3-9]|[2-6]\d)\b|\b(?:1[3-9]|[2-6]\d)\s*(?:yo|y/o|y\.o\.|years?[ -]old)\b", re.I)
URL_RE  = re.compile(r"https?://\S+|www\.\S+")
USER_RE = re.compile(r"/?u/[A-Za-z0-9_-]+")

def parse_prompt(prompt):
    m = PROMPT_RE.search(prompt or "")
    if not m:
        return None
    return {"subreddit": m["sub"].strip().lower(), "title": " ".join(m["title"].split()), "post": " ".join(m["post"].split())}

def tag_to_labels(m):
    age = int(m["a1"] or m["a2"]); g = (m["g1"] or m["g2"]).lower()
    return age, ("M" if g.startswith("m") else "F")

def extract_poster(title):
    m = POSTER_TITLE_RE.search(title)
    return tag_to_labels(m) if m else None

def body_agreement(post, age, gender):
    m = POSTER_POST_RE.search(post)
    if not m:
        return None
    a, g = tag_to_labels(m)
    return int(abs(a - age) <= 1 and g == gender)

def clean_post(post):
    t = ANY_TAG_RE.sub("", post)
    t = BARE_TAG_RE.sub("", t)
    t = URL_RE.sub("[URL]", t)
    t = USER_RE.sub("[USER]", t)
    t = re.sub(r"\s+([,.!?;:])", r"\1", t)
    return " ".join(t.split())

def truncate_words(text, max_words=60):
    sents = re.split(r"(?<=[.!?])\s+", text)
    out = []
    for s in sents:
        if len(" ".join(out + [s]).split()) > max_words:
            break
        out.append(s)
    return " ".join(out) if out else " ".join(text.split()[:max_words])

def build_reddit_dataset(records, min_words=15, max_words=60):
    funnel = {"posts parsed": len(records)}
    rows = []
    for r in records:
        lab = extract_poster(r["title"])
        if lab is None:
            continue
        age, gender = lab
        rows.append({**r, "age": age, "gender": gender,
                     "age_bracket": "<=25" if age <= 25 else ">25",
                     "body_agrees": body_agreement(r["post"], age, gender)})
    funnel["poster tag in title"] = len(rows)
    df = pd.DataFrame(rows)
    if df.empty:
        return df, funnel
    df["text"] = df["post"].map(clean_post)
    df["explicit_age"] = df["text"].str.contains(EXPLICIT_AGE_RE)
    df = df[~df["explicit_age"]]
    funnel["no explicit age in body"] = len(df)
    df["text"] = df["text"].map(lambda t: truncate_words(t, max_words))
    df["n_words"] = df["text"].str.split().str.len()
    df = df[df["n_words"] >= min_words]
    funnel[f"{min_words}-{max_words} words"] = len(df)
    df = df.drop_duplicates("text").reset_index(drop=True)
    funnel["deduplicated"] = len(df)
    return df, funnel

def balanced_sample(df, n, seed=SEED):
    cells = df.groupby(["gender", "age_bracket"])
    per = n // max(cells.ngroups, 1)
    parts = [g.sample(min(len(g), per), random_state=seed) for _, g in cells]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

In [ ]:
# ---------- Collection ----------
def load_tldr_records():
    for name in ["trl-lib/tldr", "CarperAI/openai_summarize_tldr"]:
        try:
            dsd = load_dataset(name)
            recs = []
            for split in dsd:
                for p in dsd[split]["prompt"]:
                    r = parse_prompt(p)
                    if r: recs.append(r)
            print(f"Loaded {name}: {len(recs):,} posts across splits {list(dsd.keys())}")
            return recs, name
        except Exception as ex:
            print(f"Could not load {name}: {ex}")
    raise RuntimeError("No Reddit TL;DR source could be loaded")

records, REDDIT_SOURCE = load_tldr_records()
reddit_all, funnel = build_reddit_dataset(records)
print(pd.Series(funnel).to_string())

In [ ]:
# ---------- Name sanitisation (spaCy NER) + balanced sample ----------
import spacy
nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer"])

reddit = balanced_sample(reddit_all, N_REDDIT)
def mask_names(texts):
    out = []
    for doc in nlp.pipe(texts, batch_size=256):
        t = doc.text
        for ent in reversed(doc.ents):
            if ent.label_ == "PERSON":
                t = t[:ent.start_char] + "[NAME]" + t[ent.end_char:]
        out.append(t)
    return out
reddit["text"] = mask_names(reddit["text"].tolist())
funnel["balanced sample"] = len(reddit)

# utility label: subreddit (small subreddits grouped as 'other')
top = reddit["subreddit"].value_counts()
keep = top[top >= max(50, int(0.05 * len(reddit)))].index
reddit["sub_label"] = np.where(reddit["subreddit"].isin(keep), reddit["subreddit"], "other")

reddit.to_csv(f"{OUT}/reddit_selfdisclosure_dataset.csv", index=False)
pd.Series(funnel).to_csv(f"{OUT}/reddit_funnel.csv")

agree = reddit_all["body_agrees"].dropna()
print(f"Source: {REDDIT_SOURCE}")
print(f"Full extracted set: {len(reddit_all):,} posts | experiment sample: {len(reddit):,}")
print(f"Title-vs-body label agreement (where the body repeats the tag): {agree.mean():.3f} on {len(agree):,} posts")
print("\nGender:\n", reddit_all.gender.value_counts(normalize=True).round(3).to_string())
print("\nAge bracket:\n", reddit_all.age_bracket.value_counts(normalize=True).round(3).to_string())
print(f"\nAge: mean {reddit_all.age.mean():.1f}, median {reddit_all.age.median():.0f}, range {reddit_all.age.min()}-{reddit_all.age.max()}")
print("\nSubreddit (utility label) in the sample:\n", reddit.sub_label.value_counts().to_string())
print(f"\nText length in the sample: mean {reddit.n_words.mean():.1f} words")
display(reddit[["subreddit", "title", "text", "age", "gender", "age_bracket"]].head(5))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
pd.Series(funnel).plot.barh(ax=axes[0], color="#4C72B0"); axes[0].invert_yaxis(); axes[0].set_title("Construction funnel (posts)")
reddit_all.age.clip(upper=50).plot.hist(bins=range(13, 52), ax=axes[1], color="#55A868"); axes[1].axvline(25.5, color="k", ls="--"); axes[1].set_title("Self-disclosed age (all extracted)")
reddit.sub_label.value_counts().plot.bar(ax=axes[2], color="#C44E52"); axes[2].set_title("Subreddit in sample")
plt.tight_layout(); plt.savefig(f"{OUT}/reddit_dataset_overview.png", dpi=160); plt.show()

# manual audit file (fill audit_age_ok / audit_gender_ok with 1/0, then re-upload)
# The annotator reads title + post and writes down the POSTER's gender (M/F) and age bracket (<=25 / >25),
# or "?" if it cannot be decided. The regex labels are deliberately NOT shown in the annotation file,
# so the human label is an independent second judgement (Cohen's kappa is computed in Part 6).
audit = reddit_all.sample(min(100, len(reddit_all)), random_state=SEED)[["subreddit", "title", "post", "age", "gender", "age_bracket"]].reset_index(drop=True)
audit.insert(0, "audit_id", range(len(audit)))
audit.to_csv(f"{OUT}/reddit_audit_sample_100_with_labels.csv", index=False)      # key (do not open while annotating)
blind = audit[["audit_id", "subreddit", "title", "post"]].copy()
blind["human_gender"] = ""; blind["human_age_bracket"] = ""; blind["notes"] = ""
blind.to_csv(f"{OUT}/reddit_audit_sample_100.csv", index=False)
print("Audit sample written to results/reddit_audit_sample_100.csv")

## Part 5 · Privacy experiment on the constructed dataset
Following the paper's setup:
* **Utility task:** predict the subreddit. **Privacy attacks:** predict *gender* and *age bracket*.
* **Static attacker:** trained on original text, tested on privatised text. **Adaptive attacker:** trained on privatised text, tested on privatised text.
* Classifier: TF-IDF (1–2-grams) + logistic regression. This is a fast stand-in for the paper's DeBERTaV3-base, which is the next step.
* **PP+** = accuracy − majority-class rate. **Relative Gain** RG = U_r/U_o − P_r/P_o (macro-F1). Positive means privacy improved by more than utility was lost.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split

def make_clf():
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=50000),
                         LogisticRegression(max_iter=2000, class_weight="balanced"))

TASKS = {"utility: subreddit": "sub_label", "privacy: gender": "gender", "privacy: age bracket": "age_bracket"}
if reddit["sub_label"].nunique() < 2:
    TASKS.pop("utility: subreddit"); print("Only one subreddit class - utility task skipped")

strat = reddit["gender"] + reddit["age_bracket"]
train_df, test_df = train_test_split(reddit, test_size=N_PRIV_TEST, stratify=strat, random_state=SEED)
adapt_df = train_df.sample(min(N_PRIV_ADAPT, len(train_df)), random_state=SEED)
print(f"train (original) {len(train_df)} | adaptive train (privatised) {len(adapt_df)} | test {len(test_df)}")

t = time.time()
test_priv  = {e: privatize_many(test_df["text"], e, "reddit-test") for e in EPSILONS}
adapt_priv = {e: privatize_many(adapt_df["text"], e, "reddit-adapt") for e in EPSILONS}
print(f"Privatisation done in {(time.time()-t)/60:.1f} min")
pd.DataFrame([{"epsilon": e, "original": o, "privatized": p} for e in EPSILONS
              for o, p in zip(test_df["text"], test_priv[e])]).to_csv(f"{OUT}/reddit_test_privatized.csv", index=False)

def score(clf, X, y):
    pred = clf.predict(list(X)); y = list(y)
    maj = pd.Series(y).value_counts(normalize=True).iloc[0]
    return {"F1": f1_score(y, pred, average="macro"), "Acc": accuracy_score(y, pred), "PP+": accuracy_score(y, pred) - maj}

rows = []
for task, col in TASKS.items():
    full = make_clf().fit(train_df["text"], train_df[col])    # static attacker / baseline
    small = make_clf().fit(adapt_df["text"], adapt_df[col])   # adaptive-size baseline
    rows.append({"task": task, "setting": "static", "epsilon": "original", **score(full, test_df["text"], test_df[col])})
    rows.append({"task": task, "setting": "adaptive", "epsilon": "original", **score(small, test_df["text"], test_df[col])})
    for e in EPSILONS:
        rows.append({"task": task, "setting": "static", "epsilon": e, **score(full, test_priv[e], test_df[col])})
        adapt = make_clf().fit(adapt_priv[e], adapt_df[col])
        rows.append({"task": task, "setting": "adaptive", "epsilon": e, **score(adapt, test_priv[e], test_df[col])})
attack = pd.DataFrame(rows)
attack[["F1", "Acc", "PP+"]] = (attack[["F1", "Acc", "PP+"]] * 100).round(2)
attack.to_csv(f"{OUT}/reddit_attack_results.csv", index=False)
display(attack.pivot_table(index=["task", "setting"], columns="epsilon", values="F1", sort=False))

In [ ]:
# ---------- Relative Gain + figure ----------
rg_rows = []
if "utility: subreddit" in TASKS:
    for setting in ["static", "adaptive"]:
        a = attack[attack.setting == setting].set_index(["task", "epsilon"])["F1"]
        Uo = a[("utility: subreddit", "original")]
        for priv_task in ["privacy: gender", "privacy: age bracket"]:
            Po = a[(priv_task, "original")]
            for e in EPSILONS:
                rg_rows.append({"setting": setting, "attribute": priv_task.split(": ")[1], "epsilon": e,
                                "U_r/U_o": a[("utility: subreddit", e)] / Uo, "P_r/P_o": a[(priv_task, e)] / Po,
                                "RG": a[("utility: subreddit", e)] / Uo - a[(priv_task, e)] / Po})
rg = pd.DataFrame(rg_rows).round(3)
rg.to_csv(f"{OUT}/reddit_relative_gain.csv", index=False)
if len(rg): display(rg.pivot_table(index=["setting", "attribute"], columns="epsilon", values="RG", sort=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, setting in zip(axes, ["static", "adaptive"]):
    for task in TASKS:
        r = attack[(attack.setting == setting) & (attack.task == task)]
        line, = ax.plot(EPSILONS, r[r.epsilon != "original"]["F1"], marker="o", label=task)
        ax.axhline(r[r.epsilon == "original"]["F1"].iloc[0], ls="--", lw=0.8, color=line.get_color())
    ax.set_xscale("log"); ax.set_xticks(EPSILONS); ax.set_xticklabels(EPSILONS)
    ax.set_title(f"{setting} attacker (macro-F1)"); ax.set_xlabel("ε (per token)"); ax.grid(alpha=0.3)
axes[0].legend(fontsize=8); plt.tight_layout(); plt.savefig(f"{OUT}/reddit_privacy_utility.png", dpi=160); plt.show()

## Part 6 · LLM-as-judge, LLM annotator and LLM attacker
An instruction-tuned open LLM (`Qwen/Qwen2.5-3B-Instruct`, fp16 on the T4, greedy decoding) is used in three roles:

1. **Judge of rewrite quality.** For a sample of (original, rewritten) pairs from SST-2 and Reddit at ε ∈ {10, 50, 250}, the LLM rates **meaning preservation** and **fluency** on a 1–5 scale. Two control conditions check the judge: *identical* pairs (should get 5) and *mismatched* pairs, where the rewrite is replaced by a random other text (meaning should get ≈1). We also report the Spearman correlation between the judge's meaning score and SBERT cosine similarity.
2. **Second annotator for the constructed labels.** On the 100-post audit sample the LLM reads title + post and states the poster's gender and age; Cohen's κ against the regex labels (and against the human audit, if filled in) measures label agreement.
3. **Zero-shot attacker.** The LLM is asked to guess gender and age bracket from original and privatised Reddit test posts, a stronger, more "human-like" adversary than TF-IDF.

In [ ]:
# ---------- Load the judge LLM ----------
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score

JUDGE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
jtok = AutoTokenizer.from_pretrained(JUDGE_MODEL, padding_side="left")
jlm = AutoModelForCausalLM.from_pretrained(JUDGE_MODEL, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
                                           device_map="auto" if DEVICE == "cuda" else None)
jlm.eval()

def llm_generate(prompts, system, max_new_tokens=16, batch_size=16):
    outs = []
    for i in tqdm(range(0, len(prompts), batch_size), desc="LLM", leave=False):
        chats = [jtok.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": p}],
                                          tokenize=False, add_generation_prompt=True) for p in prompts[i:i + batch_size]]
        enc = jtok(chats, return_tensors="pt", padding=True).to(jlm.device)
        with torch.no_grad():
            gen = jlm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=jtok.eos_token_id)
        outs += jtok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

JUDGE_SYSTEM = "You are a careful annotator who evaluates rewritten English text. Answer only in the requested format."
JUDGE_TMPL = """Text A is an original text. Text B is an automatically rewritten version of it.

Text A: {a}
Text B: {b}

Rate Text B on two 1-5 scales.
Meaning: how well does B preserve the meaning of A? (1 = unrelated, 3 = partly preserved, 5 = same meaning)
Fluency: how natural and grammatical is B as English, ignoring A? (1 = gibberish, 3 = understandable but awkward, 5 = fully fluent)
Answer exactly as: Meaning: <1-5>; Fluency: <1-5>"""

def parse_judge(s):
    m = re.search(r"Meaning:\s*([1-5]).*?Fluency:\s*([1-5])", s, re.S | re.I)
    return (int(m[1]), int(m[2])) if m else (np.nan, np.nan)
print("Judge loaded:", JUDGE_MODEL)

In [ ]:
# ---------- Role 1: LLM-as-judge of rewrite quality ----------
N_JUDGE = 40 if QUICK else 100
JUDGE_EPS = [10, 50, 250]
rng = np.random.RandomState(SEED)
pools = {"SST-2": (list(sst["sentence"][:N_JUDGE]), {e: sst_priv[e][:N_JUDGE] for e in EPSILONS}),
         "Reddit": (list(test_df["text"][:N_JUDGE]), {e: test_priv[e][:N_JUDGE] for e in EPSILONS})}
jrows = []
for ds, (orig, priv) in pools.items():
    for i, o in enumerate(orig):
        jrows.append({"dataset": ds, "condition": "identical (control)", "id": i, "a": o, "b": o})
        j = (i + 1 + rng.randint(len(orig) - 1)) % len(orig)
        jrows.append({"dataset": ds, "condition": "mismatched (control)", "id": i, "a": o, "b": orig[j]})
        for e in JUDGE_EPS:
            jrows.append({"dataset": ds, "condition": f"eps={e}", "id": i, "a": o, "b": priv[e][i]})
judge = pd.DataFrame(jrows)
raw = llm_generate([JUDGE_TMPL.format(a=a, b=b) for a, b in zip(judge.a, judge.b)], JUDGE_SYSTEM, max_new_tokens=16)
judge["raw"] = raw
judge[["meaning", "fluency"]] = [parse_judge(r) for r in raw]
judge["cossim"] = [float((sbert.encode(a, normalize_embeddings=True) * sbert.encode(b, normalize_embeddings=True)).sum())
                   for a, b in zip(judge.a, judge.b)]
judge.to_csv(f"{OUT}/llm_judge_scores.csv", index=False)

order = ["identical (control)"] + [f"eps={e}" for e in JUDGE_EPS] + ["mismatched (control)"]
judge_tab = (judge.groupby(["dataset", "condition"])[["meaning", "fluency", "cossim"]].mean()
             .reindex(pd.MultiIndex.from_product([pools.keys(), order])).round(2))
judge_parse_rate = judge["meaning"].notna().mean()
rw = judge[judge.condition.str.startswith("eps")].dropna(subset=["meaning"])
judge_rho, judge_p = spearmanr(rw["meaning"], rw["cossim"])
print(f"parse rate {judge_parse_rate:.3f} | Spearman(meaning, cosine) on rewrites = {judge_rho:.3f} (p={judge_p:.1e}, n={len(rw)})")
display(judge_tab)

In [ ]:
# ---------- Role 2: LLM as second annotator of the constructed labels (Cohen's kappa) ----------
ANN_SYSTEM = "You extract information that a Reddit poster states about themselves. Answer only in the requested format."
ANN_TMPL = """Reddit title: {t}
Reddit post: {p}

What gender and age does the POSTER (the person who wrote this, not their partner or anyone else) state for themselves?
Answer exactly as: Gender: <M or F or ?>; Age: <number or ?>"""

def parse_ann(s):
    g = re.search(r"Gender:\s*([MF?])", s, re.I); a = re.search(r"Age:\s*(\d{1,2})", s)
    gender = g[1].upper() if g and g[1] != "?" else "?"
    bracket = ("<=25" if int(a[1]) <= 25 else ">25") if a else "?"
    return gender, bracket

key = pd.read_csv(f"{OUT}/reddit_audit_sample_100_with_labels.csv")
ann_raw = llm_generate([ANN_TMPL.format(t=t, p=p[:1500]) for t, p in zip(key.title, key.post)], ANN_SYSTEM, max_new_tokens=16)
key[["llm_gender", "llm_age_bracket"]] = [parse_ann(r) for r in ann_raw]

def kappa(a, b):
    a, b = list(a), list(b)
    return cohen_kappa_score(a, b), float(np.mean(np.array(a) == np.array(b)))

agreement = []
for attr, reg, llm in [("gender", "gender", "llm_gender"), ("age bracket", "age_bracket", "llm_age_bracket")]:
    k, acc = kappa(key[reg], key[llm]); agreement.append({"attribute": attr, "pair": "regex vs LLM", "n": len(key), "kappa": k, "raw agreement": acc})

# Human audit: fill human_gender / human_age_bracket in results/reddit_audit_sample_100.csv
# (open it in Google Sheets, fill in, download as CSV, upload to results/ with the same name), then re-run this cell.
human = pd.read_csv(f"{OUT}/reddit_audit_sample_100.csv").merge(key, on=["audit_id", "subreddit", "title", "post"])
human = human[human["human_gender"].astype(str).str.strip().isin(["M", "F", "?"])]
N_HUMAN = len(human)
if N_HUMAN:
    for attr, reg, llm, hum in [("gender", "gender", "llm_gender", "human_gender"), ("age bracket", "age_bracket", "llm_age_bracket", "human_age_bracket")]:
        h = human[hum].astype(str).str.strip()
        for name, other in [("regex vs human", human[reg]), ("LLM vs human", human[llm])]:
            k, acc = kappa(other, h); agreement.append({"attribute": attr, "pair": name, "n": N_HUMAN, "kappa": k, "raw agreement": acc})
else:
    print("Human audit not filled in yet - only regex vs LLM agreement is reported.")
agreement = pd.DataFrame(agreement).round(3)
agreement.to_csv(f"{OUT}/label_agreement.csv", index=False)
key.to_csv(f"{OUT}/reddit_audit_llm_labels.csv", index=False)
display(agreement)

In [ ]:
# ---------- Role 3: zero-shot LLM attacker on the Reddit test posts ----------
N_LLM_ATT = 100 if QUICK else N_PRIV_TEST
ATT_SYSTEM = "You are a profiling model. Always give your best guess, never refuse. Answer only in the requested format."
ATT_TMPL = """Here is a Reddit post with all explicit age and gender tags removed:

{p}

Guess the author's gender and whether the author is 25 or younger.
Answer exactly as: Gender: <M or F>; Age: <25 or younger / over 25>"""

def parse_att(s):
    g = re.search(r"Gender:\s*([MF])", s, re.I)
    young = re.search(r"25 or younger|younger|<=\s*25", s, re.I)
    old = re.search(r"over 25|older|>\s*25", s, re.I)
    return (g[1].upper() if g else "M"), ("<=25" if (young and not old) else ">25")

sub = test_df.iloc[:N_LLM_ATT]
conds = {"original": list(sub["text"])} | {e: test_priv[e][:N_LLM_ATT] for e in [10, 50, 250]}
llm_att = []
for cond, texts in conds.items():
    preds = [parse_att(r) for r in llm_generate([ATT_TMPL.format(p=t) for t in texts], ATT_SYSTEM, max_new_tokens=20)]
    for attr, col, k in [("gender", "gender", 0), ("age bracket", "age_bracket", 1)]:
        y = list(sub[col]); p = [x[k] for x in preds]
        llm_att.append({"attribute": attr, "epsilon": cond, "F1": 100 * f1_score(y, p, average="macro"), "Acc": 100 * accuracy_score(y, p)})
llm_att = pd.DataFrame(llm_att).round(2)
llm_att.to_csv(f"{OUT}/reddit_llm_attacker.csv", index=False)
display(llm_att.pivot_table(index="attribute", columns="epsilon", values="F1", sort=False))

## Part 7 · Export for the report
Writes `results/generated/` with `numbers.tex` (single values as LaTeX macros), the LaTeX tables and the figures used in the report. Upload the **contents** of `generated/` into the `generated/` folder of the Overleaf project (overwrite the placeholders) and recompile.

In [ ]:
# ---------- Export numbers / tables / figures for the LaTeX report ----------
import platform, transformers, sentence_transformers, sklearn, importlib.metadata as im
GEN = f"{OUT}/generated"; os.makedirs(GEN, exist_ok=True)
EPS_TEX = r"$\varepsilon$="

def tex(s):
    s = str(s)
    rep = {"\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$", "#": r"\#", "_": r"\_", "{": r"\{",
           "}": r"\}", "~": r"\textasciitilde{}", "^": r"\textasciicircum{}", "<": r"$<$", ">": r"$>$"}
    return "".join(rep.get(c, c) for c in s)
def f2(x, d=2): return "--" if pd.isna(x) else f"{x:.{d}f}"

macros = {}
def mac(name, val): macros[name] = str(val)

# --- environment + mechanism ---
mac("EnvGPU", tex(torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"))
mac("EnvTorch", tex(torch.__version__)); mac("EnvTransformers", tex(transformers.__version__))
mac("EnvPython", tex(platform.python_version())); mac("EnvDpmlm", tex(im.version("dpmlm")))
mac("EnvSbert", tex(sentence_transformers.__version__)); mac("EnvSacrebleu", tex(sacrebleu.__version__))
mac("ClipMin", f"{M.clip_min:.2f}"); mac("ClipMax", f"{M.clip_max:.2f}"); mac("Sensitivity", f"{M.sensitivity:.2f}")
mac("NSST", len(sst)); mac("NIMDB", len(imdb_text)); mac("SSTLen", f"{np.mean([len(s.split()) for s in sst['sentence']]):.1f}")
mac("IMDBFullLen", f"{np.mean(imdb_full_len):.0f}"); mac("IMDBCutLen", f"{np.mean([len(t.split()) for t in imdb_text]):.0f}")

# --- Reddit dataset statistics ---
mac("RedditSource", tex(REDDIT_SOURCE))
fkeys = list(funnel.keys())
for name, k in zip(["FunParsed", "FunTag", "FunNoAge", "FunLen", "FunDedup", "FunBalanced"], fkeys):
    mac(name, f"{funnel[k]:,}".replace(",", "{,}"))
mac("AgreeRate", f"{100 * agree.mean():.1f}"); mac("AgreeN", f"{len(agree):,}".replace(",", "{,}"))
mac("PctFemale", f"{100 * (reddit_all.gender == 'F').mean():.1f}")
mac("PctYoung", f"{100 * (reddit_all.age_bracket == '<=25').mean():.1f}")
mac("AgeMedian", f"{reddit_all.age.median():.0f}"); mac("AgeMin", int(reddit_all.age.min())); mac("AgeMax", int(reddit_all.age.max()))
mac("PctMinor", f"{100 * (reddit_all.age < 18).mean():.1f}")
mac("NSubLabels", reddit["sub_label"].nunique()); mac("SubLabels", tex(", ".join(sorted(reddit["sub_label"].unique()))))
mac("RedditLen", f"{reddit.n_words.mean():.1f}")
mac("NTrain", len(train_df)); mac("NAdapt", len(adapt_df)); mac("NTest", len(test_df))

# --- judge / agreement ---
mac("JudgeModel", tex(JUDGE_MODEL)); mac("NJudge", N_JUDGE); mac("JudgeParse", f"{100 * judge_parse_rate:.1f}")
mac("JudgeRho", f"{judge_rho:.2f}"); mac("JudgeRhoN", len(rw)); mac("NHuman", N_HUMAN); mac("NLLMAtt", N_LLM_ATT)
for _, r in agreement.iterrows():
    nm = {"regex vs LLM": "RegLLM", "regex vs human": "RegHum", "LLM vs human": "LLMHum"}[r.pair] + ("G" if r.attribute == "gender" else "A")
    mac("Kappa" + nm, f"{r.kappa:.2f}"); mac("Raw" + nm, f"{100 * r['raw agreement']:.0f}")

# --- headline numbers used in the text ---
def U(res, e, col): return float(res[res.epsilon.astype(str) == str(e)][col].iloc[0])
for ds, res in [("SST", sst_res), ("IMDB", imdb_res)]:
    for e in ["original"] + EPSILONS:
        tag = "Orig" if e == "original" else {10: "Ten", 25: "TwentyFive", 50: "Fifty", 100: "Hundred", 250: "TwoFifty"}[e]
        mac(f"{ds}Acc{tag}", f"{100 * U(res, e, 'Acc'):.1f}")
        mac(f"{ds}Bleu{tag}", f"{U(res, e, 'BLEU') / 100:.2f}"); mac(f"{ds}Cos{tag}", f"{U(res, e, 'CosSim'):.2f}")
        mac(f"{ds}Chg{tag}", f"{U(res, e, '%words changed'):.0f}")

with open(f"{GEN}/numbers.tex", "w") as f:
    f.write("% auto-generated by the notebook - do not edit by hand\n")
    for k, v in macros.items(): f.write(f"\\newcommand{{\\{k}}}{{{v}}}\n")

# --- Table: SST-2 replication vs paper ---
PAPER = {"Acc": [68.50, 76.49, 84.10, 86.16, 86.05], "BLEU": [0.08, 0.11, 0.19, 0.22, 0.23], "CS": [0.17, 0.37, 0.61, 0.65, 0.67]}
L = [r"\begin{tabular}{llrrrrrr}", r"\toprule", r" & & Orig. & $\varepsilon$=10 & 25 & 50 & 100 & 250 \\", r"\midrule"]
def row(lbl, src, vals): return f"{lbl} & {src} & " + " & ".join(vals) + r" \\"
L.append(row("BLEU", "paper", ["--"] + [f2(v) for v in PAPER["BLEU"]]))
L.append(row("", "ours", [f2(1.0)] + [f2(U(sst_res, e, "BLEU") / 100) for e in EPSILONS]))
L.append(row("CosSim", "paper", ["--"] + [f2(v) for v in PAPER["CS"]]))
L.append(row("", "ours", [f2(1.0)] + [f2(U(sst_res, e, "CosSim")) for e in EPSILONS]))
L.append(row(r"Acc (\%)", r"paper$^\dagger$", ["95.72"] + [f2(v, 1) for v in PAPER["Acc"]]))
L.append(row("", r"ours$^\ddagger$", [f2(100 * U(sst_res, e, "Acc"), 1) for e in ["original"] + EPSILONS]))
L.append(row(r"\% changed", "ours", [f2(U(sst_res, e, "%words changed"), 0) for e in ["original"] + EPSILONS]))
L += [r"\bottomrule", r"\end{tabular}"]
open(f"{GEN}/tab_sst2.tex", "w").write("\n".join(L))

# --- Table: utility on SST-2 vs IMDb ---
L = [r"\begin{tabular}{llrrrrrr}", r"\toprule", r" & & Orig. & 10 & 25 & 50 & 100 & 250 \\", r"\midrule"]
for metric, lbl, scale, d in [("BLEU", "BLEU", 0.01, 2), ("CosSim", "CosSim", 1, 2), ("Acc", r"Acc (\%)", 100, 1), ("%words changed", r"\% changed", 1, 0)]:
    for ds, res in [("SST-2", sst_res), ("IMDb", imdb_res)]:
        L.append(row(lbl if ds == "SST-2" else "", ds, [f2(scale * U(res, e, metric), d) for e in ["original"] + EPSILONS]))
L += [r"\bottomrule", r"\end{tabular}"]
open(f"{GEN}/tab_imdb.tex", "w").write("\n".join(L))

# --- Table: construction funnel ---
L = [r"\begin{tabular}{lr}", r"\toprule", r"Step & Posts \\", r"\midrule"]
L += [f"{tex(k)} & {v:,} \\\\".replace(",", "{,}") for k, v in funnel.items()]
L += [r"\bottomrule", r"\end{tabular}"]
open(f"{GEN}/tab_funnel.tex", "w").write("\n".join(L))

# --- Table: Reddit attacks (TF-IDF static/adaptive + LLM) and RG ---
cols = ["original"] + EPSILONS
L = [r"\begin{tabular}{llrrrrrr}", r"\toprule", r"Task & Attacker & Orig. & 10 & 25 & 50 & 100 & 250 \\", r"\midrule"]
for task in TASKS:
    for setting in ["static", "adaptive"]:
        a = attack[(attack.task == task) & (attack.setting == setting)].set_index("epsilon")["F1"]
        name = "Subreddit" if task.startswith("utility") else tex(task.split(": ")[1]).capitalize()
        L.append(f"{name if setting == 'static' else ''} & {setting} & " + " & ".join(f2(a.get(c, np.nan), 1) for c in cols) + r" \\")
    if task.startswith("privacy"):
        attr = task.split(": ")[1]
        b = llm_att[llm_att.attribute == attr].set_index("epsilon")["F1"]
        L.append(f" & LLM zero-shot & " + " & ".join(f2(b.get(c, np.nan), 1) for c in cols) + r" \\")
    L.append(r"\midrule")
for setting in ["static", "adaptive"]:
    for attr in ["gender", "age bracket"]:
        r = rg[(rg.setting == setting) & (rg.attribute == attr)].set_index("epsilon")["RG"] if len(rg) else pd.Series(dtype=float)
        L.append(f"RG {tex(attr)} & {setting} & -- & " + " & ".join(f2(r.get(e, np.nan), 2) for e in EPSILONS) + r" \\")
L += [r"\bottomrule", r"\end{tabular}"]
open(f"{GEN}/tab_reddit.tex", "w").write("\n".join(L))

# --- Table: LLM judge ---
L = [r"\begin{tabular}{llrrr}", r"\toprule", r"Data & Condition & Meaning & Fluency & CosSim \\", r"\midrule"]
for (ds, cond), r in judge_tab.iterrows():
    cond_tex = tex(cond).replace("eps=", EPS_TEX)
    L.append(f"{ds if cond == order[0] else ''} & {cond_tex} & {f2(r.meaning)} & {f2(r.fluency)} & {f2(r.cossim)} \\\\")
    if cond == order[-1] and ds != list(pools)[-1]: L.append(r"\midrule")
L += [r"\bottomrule", r"\end{tabular}"]
open(f"{GEN}/tab_judge.tex", "w").write("\n".join(L))

# --- Table: label agreement ---
L = [r"\begin{tabular}{llrrr}", r"\toprule", r"Attribute & Pair & $n$ & $\kappa$ & Agree (\%) \\", r"\midrule"]
for _, r in agreement.iterrows():
    L.append(f"{tex(r.attribute)} & {tex(r.pair)} & {r.n} & {r.kappa:.2f} & {100 * r['raw agreement']:.0f} \\\\")
L += [r"\bottomrule", r"\end{tabular}"]
open(f"{GEN}/tab_agreement.tex", "w").write("\n".join(L))

# --- Example rewrites (demo sentence 1 + one SST-2 sentence) ---
def ex_block(orig, rewrites):
    out = [r"\begin{tabular}{@{}p{0.11\linewidth}p{0.85\linewidth}@{}}", r"\toprule", f"Original & {tex(orig)} \\\\", r"\midrule"]
    out += [f"{EPS_TEX}{e} & {tex(r)} \\\\" for e, r in rewrites]
    return out + [r"\bottomrule", r"\end{tabular}"]
d0 = demo_df[demo_df.original == demo[0]]
open(f"{GEN}/ex_demo.tex", "w").write("\n".join(ex_block(demo[0], zip(d0.epsilon, d0.rewritten))))
i_ex = int(np.argmin([abs(len(s.split()) - 12) for s in sst["sentence"]]))
open(f"{GEN}/ex_sst2.tex", "w").write("\n".join(ex_block(sst["sentence"][i_ex], [(e, sst_priv[e][i_ex]) for e in EPSILONS])))

# --- Reddit data snippet: one cleaned example per gender x age cell (short ones) ---
L = [r"\begin{tabular}{@{}p{0.17\linewidth}p{0.79\linewidth}@{}}", r"\toprule"]
for (g, b), grp in reddit.groupby(["gender", "age_bracket"]):
    r = grp.sort_values("n_words").iloc[len(grp) // 10]
    L.append(f"\\textbf{{{g}, {tex(b)}}} r/{tex(r.subreddit)} & {tex(r.text)} \\\\"); L.append(r"\midrule")
L[-1] = r"\bottomrule"; L.append(r"\end{tabular}")
open(f"{GEN}/ex_reddit.tex", "w").write("\n".join(L))
r_ex = test_df.iloc[int(np.argmin([abs(len(t.split()) - 25) for t in test_df.text[:50]]))]
j_ex = list(test_df.text).index(r_ex.text)
open(f"{GEN}/ex_reddit_priv.tex", "w").write("\n".join(ex_block(r_ex.text, [(e, test_priv[e][j_ex]) for e in [10, 50, 250]])))

# --- figures ---
for fpng in ["utility_sst2_imdb.png", "reddit_dataset_overview.png", "reddit_privacy_utility.png"]:
    if os.path.exists(f"{OUT}/{fpng}"): shutil.copy(f"{OUT}/{fpng}", f"{GEN}/{fpng}")
print("Wrote:", sorted(os.listdir(GEN)))
print(open(f"{GEN}/numbers.tex").read())

## Summary (numbers for the slides)

In [ ]:
lines = ["# DP-MLM project update - results summary\n",
         f"Device: {DEVICE} | QUICK={QUICK} | eps={EPSILONS}\n",
         "## Utility on original + existing data\n", pd.concat([sst_res, imdb_res]).to_markdown(index=False), "\n",
         "## Constructed Reddit dataset - construction funnel\n", pd.Series(funnel).to_frame("posts").to_markdown(), "\n",
         f"Title-vs-body label agreement: {agree.mean():.3f} (n={len(agree)})\n",
         "## Attack results (macro-F1 %)\n",
         attack.pivot_table(index=["task", "setting"], columns="epsilon", values="F1", sort=False).to_markdown(), "\n"]
if len(rg):
    lines += ["## Relative Gain\n", rg.pivot_table(index=["setting", "attribute"], columns="epsilon", values="RG", sort=False).to_markdown()]
open(f"{OUT}/summary.md", "w").write("\n".join(lines))
print("\n".join(lines))

lines += ["\n## LLM-as-judge\n", judge_tab.to_markdown(), "\n## Label agreement (Cohen's kappa)\n", agreement.to_markdown(index=False),
          "\n## Zero-shot LLM attacker (macro-F1 %)\n", llm_att.pivot_table(index="attribute", columns="epsilon", values="F1", sort=False).to_markdown()]
open(f"{OUT}/summary.md", "w").write("\n".join(lines))
with zipfile.ZipFile("results.zip", "w") as z:
    for root, _, files_ in os.walk(OUT):
        for f in files_: z.write(os.path.join(root, f))
try:
    from google.colab import files; files.download("results.zip")
except Exception:
    print("results.zip written")